# Regional Obligations Profiling
Owner: Joy
Table: `pondolens.dev_bronze.regional_expenditure_obligations`
Baseline: 1,785 rows

## Goal
Identify data issues and propose Silver treatments. Do not modify Bronze.

## Checks
- Check counts by file/year, column types, nulls/blanks, and `_rescued_data`.
- Define what one row represents; check exact duplicates and repeated business keys.
- Verify regions, years, categories, particulars, numeric amounts, units, and price basis.
- Confirm reporting periods and actual/estimate labels.
- Identify totals/subtotals to avoid double counting.
- Compare sample amounts and totals with the original source.
- With Anji, check whether obligations and GAA have matching scope, periods, and regions.
- Check regional compatibility with population and GRDP.

## Findings
| Check / Issue | Affected rows / Evidence | Proposed Silver treatment |
|---|---|---|
| To be completed | | |

## Assumptions to Verify
| Interpretation / Question | Evidence | Status |
|---|---|---|
| To be completed | | Pending |

## Semantic Notes
Document the obligation measure, unit conversion, and rows safe to sum.
Obligations are not automatically cash payments.
A GAA–obligations difference is not automatically unobligated allotments.

## Conclusion
- Required Silver changes:
- Ready for regional comparison? Why / why not:
- Open questions:

## 1. Setup
Load the Bronze table (read-only) and map logical fields to actual column names in `COLS`. Adjust after seeing the schema.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

TABLE = "pondolens.dev_bronze.regional_expenditure_obligations"
df = spark.table(TABLE)

# Adjust after seeing the schema in cell 2. Set to None if a column doesn't exist.
COLS = dict(
    region="region",
    year="year",
    category="category",          # e.g. sector / expense class
    particulars="particulars",
    amount="amount",
    unit="unit",
    price_basis="price_basis",    # current / constant
    period="period",              # reporting period
    status="status",              # actual / estimate / program
)
KEY_CANDIDATES = [COLS["region"], COLS["year"], COLS["category"], COLS["particulars"]]
KEY_CANDIDATES = [c for c in KEY_CANDIDATES if c]

## 2. Row count and schema
Confirm the table matches the 1,785-row baseline and review column types.

In [0]:
n = df.count()
print(f"Rows: {n:,} | Baseline: 1,785 | Match: {n == 1785}")
df.printSchema()
display(df.limit(20))

## 3. Counts by file and year
Check that every source file and year is present and row counts look reasonable.

In [0]:
file_col = next((c for c in df.columns if c.lower() in
                 ("source_file", "file_name", "_source_file", "file_path")), None)

if file_col:
    display(df.groupBy(file_col).count().orderBy(file_col))
else:
    display(df.select("_metadata.file_path").groupBy("file_path").count())

if COLS["year"]:
    display(df.groupBy(COLS["year"]).count().orderBy(COLS["year"]))

## 4. Nulls and blanks
Count nulls, empty strings, and placeholder tokens (`-`, `N/A`, `...`) per column.

In [0]:
total = df.count()
rows = []
for c, t in df.dtypes:
    is_str = t == "string"
    nulls = df.filter(F.col(c).isNull()).count()
    blanks = df.filter(F.trim(F.col(c)) == "").count() if is_str else 0
    tokens = df.filter(F.trim(F.col(c)).isin("-", "--", "N/A", "n/a", "NA", "...")).count() if is_str else 0
    rows.append((c, t, nulls, blanks, tokens, round(100 * (nulls + blanks + tokens) / total, 1)))

display(spark.createDataFrame(rows, ["column", "type", "nulls", "blanks", "placeholders", "pct_missing"])
        .orderBy(F.desc("pct_missing")))

## 5. `_rescued_data`
Rows with rescued data mean the schema didn't fit the source. Check which keys were rescued.

In [0]:
if "_rescued_data" in df.columns:
    resc = df.filter(F.col("_rescued_data").isNotNull())
    print(f"Rows with rescued data: {resc.count():,}")
    display(resc.limit(50))
    display(resc.select(F.explode(F.map_keys(F.from_json("_rescued_data", "map<string,string>"))).alias("key"))
                .groupBy("key").count().orderBy(F.desc("count")))
else:
    print("No _rescued_data column")

## 6. Grain, duplicates, repeated keys
Define what one row represents. Check exact duplicates (ignoring metadata) and repeated business keys.

In [0]:

data_cols = [c for c in df.columns if not c.startswith("_")]

exact = df.groupBy(*data_cols).count().filter("count > 1")
print(f"Exact duplicate groups: {exact.count():,}")
display(exact.orderBy(F.desc("count")))

keys = df.groupBy(*KEY_CANDIDATES).count().filter("count > 1")
print(f"Repeated key groups on {KEY_CANDIDATES}: {keys.count():,}")
display(keys.orderBy(F.desc("count")))

w = Window.partitionBy(*KEY_CANDIDATES)
display(df.withColumn("n", F.count("*").over(w)).filter("n > 1").orderBy(*KEY_CANDIDATES).limit(100))

## 7. Regions, years, categories, particulars
Review distinct values, label variants (case/whitespace), and region x year completeness.

In [0]:
for key in ("region", "year", "category", "particulars"):
    c = COLS[key]
    if c:
        print(f"--- {key} ---")
        display(df.groupBy(c).count().orderBy(c))

for key in ("region", "category", "particulars"):
    c = COLS[key]
    if c:
        norm = F.lower(F.regexp_replace(F.trim(F.col(c)), r"\s+", " "))
        display(df.groupBy(norm.alias("normalized"))
                  .agg(F.countDistinct(c).alias("n_variants"), F.collect_set(c).alias("variants"))
                  .filter("n_variants > 1"))

if COLS["region"] and COLS["year"]:
    display(df.groupBy(COLS["region"]).pivot(COLS["year"]).count().orderBy(COLS["region"]))

## 8. Amounts, units, price basis
Test that amounts are numeric, look for scale problems, and check unit and price-basis consistency.

In [0]:
amt = COLS["amount"]
casted = F.col(amt).cast("double")

bad = df.filter(F.col(amt).isNotNull() & casted.isNull())
print(f"Non-castable amounts: {bad.count():,}")
display(bad.select(amt).distinct().limit(50))

display(df.select(
    F.min(casted).alias("min"), F.max(casted).alias("max"), F.mean(casted).alias("mean"),
    F.sum((casted < 0).cast("int")).alias("negatives"),
    F.sum((casted == 0).cast("int")).alias("zeros"),
    F.percentile_approx(casted, [0.01, 0.5, 0.99]).alias("p01_p50_p99"),
))

for key in ("unit", "price_basis"):
    if COLS[key]:
        display(df.groupBy(COLS[key]).count().orderBy(F.desc("count")))
if COLS["unit"] and COLS["year"]:
    display(df.groupBy(COLS["year"], COLS["unit"]).count().orderBy(COLS["year"]))

## 9. Reporting periods and actual/estimate labels
Confirm each year's status and look for labels hidden in the particulars text.

In [0]:
for key in ("period", "status"):
    if COLS[key]:
        display(df.groupBy(COLS[key]).count().orderBy(COLS[key]))
if COLS["year"] and COLS["status"]:
    display(df.groupBy(COLS["year"], COLS["status"]).count().orderBy(COLS["year"]))

display(df.filter(F.col(COLS["particulars"]).rlike(r"(?i)(actual|estimate|projected|budget|proposed|program)")).limit(50))

## 10. Totals and subtotals
Flag total-like rows to avoid double counting, and test whether detail rows reconcile to reported totals.

In [0]:
p, r, y = COLS["particulars"], COLS["region"], COLS["year"]
total_pat = r"(?i)^\s*(total|grand total|sub-?total|sum|overall)\b|\b(total|subtotal)\s*$|\ball regions\b|\bphilippines\b|\bnational\b"

flagged = df.withColumn("looks_like_total", F.col(p).rlike(total_pat))
display(flagged.filter("looks_like_total").groupBy(p).count().orderBy(F.desc("count")))

detail = (flagged.filter(~F.col("looks_like_total")).groupBy(r, y)
          .agg(F.sum(casted).alias("detail_sum")))
totals = (flagged.filter("looks_like_total").groupBy(r, y)
          .agg(F.sum(casted).alias("reported_total")))
recon = (detail.join(totals, [r, y], "full")
         .withColumn("diff", F.col("reported_total") - F.col("detail_sum")))
display(recon.orderBy(F.desc(F.abs("diff"))))

## 11. Compare with original source
Pull a reproducible sample and region/year totals to tick against the PDF/Excel. Record page/cell as evidence.

In [0]:
display(df.orderBy(F.rand(seed=42)).limit(25))

display(df.groupBy(r, y)
          .agg(F.sum(casted).alias("bronze_total"), F.count("*").alias("rows"))
          .orderBy(r, y))

%md
## 12. Obligations vs GAA (with Ange)
Compare regions, years, and category vocabularies. Do not treat GAA minus obligations as unobligated allotments until scope, period, and units match.

In [0]:
GAA_TABLE = "pondolens.dev_bronze.<gaa_table>"  # confirm with Anji
gaa = spark.table(GAA_TABLE)

def vals(d, c): return {x[0] for x in d.select(c).distinct().collect()}

for label, oc, gc in [("region", COLS["region"], "region"), ("year", COLS["year"], "year")]:
    o, g = vals(df, oc), vals(gaa, gc)  # adjust GAA column names
    print(f"{label} | obligations only: {sorted(o - g)} | GAA only: {sorted(g - o)}")

display(df.select(COLS["category"]).distinct())
display(gaa.select("category").distinct())  # adjust column

## 13. Compatibility with population and GRDP
Check region labels and year coverage. Watch for NCR, CAR, CALABARZON, MIMAROPA, BARMM, Negros Island Region, and "Nationwide / Not Regionalized" rows.

In [0]:
POP_TABLE  = "pondolens.dev_bronze.<population_table>"
GRDP_TABLE = "pondolens.dev_bronze.<grdp_table>"
pop, grdp = spark.table(POP_TABLE), spark.table(GRDP_TABLE)

for name, other in [("Population", pop), ("GRDP", grdp)]:
    o, b = vals(df, COLS["region"]), vals(other, "region")  # adjust column
    print(f"== Obligations vs {name} ==")
    print("Only in obligations:", sorted(o - b))
    print("Only in other:", sorted(b - o))

for name, t, yc in [("obligations", df, COLS["year"]), ("population", pop, "year"), ("grdp", grdp, "year")]:
    print(name, sorted(vals(t, yc)))

## Summary
Dataset: `pondolens.dev_bronze.regional_expenditure_obligations` — 1,785 rows (baseline), profiled read-only.

### What was checked
1–4. Row count vs baseline, schema, per-file/per-year counts, nulls/blanks/placeholders, `_rescued_data`.
5–7. Grain, exact duplicates, repeated business keys (region × year × category × particulars), label variants, region × year completeness.
8–9. Amount castability, scale, negatives/zeros; unit and price-basis consistency; period and actual/estimate labels.
10–11. Total/subtotal rows flagged to prevent double counting; detail-vs-reported reconciliation.
12–13. Sample and region/year totals for source tick-out; scope vs GAA (with Anji); region/year compatibility with population and GRDP.

### Key findings
- Counts / baseline: <fill in after run>
- Duplicates / rescued data: <fill in>
- Amounts, units, price basis: <fill in>
- Totals vs details: <fill in>

### Conclusion
- Required Silver treatments: <fill in>
- Ready for regional comparison: <yes/no — why>
- Open questions (for Anji / source verification): <fill in>

Obligations are not automatically cash payments; a GAA–obligations difference is not automatically unobligated allotments.